In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!unzip -q "/content/drive/MyDrive/KhPI_HW/linear_algebra_intro_practice-main.zip" -d /content/idpp
!find /content/idpp -name "matrices.py"
!find /content/idpp -name "pyproject.toml"

/content/idpp/linear_algebra_intro_practice-main/matrices.py
/content/idpp/linear_algebra_intro_practice-main/pyproject.toml


In [3]:
%cd /content/idpp/linear_algebra_intro_practice-main

/content/idpp/linear_algebra_intro_practice-main


In [4]:
!pip -q install uv
!uv sync
!uv run pytest tests/test_matrices.py -v

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 79.2 MB/s eta 0:00:00
Using CPython 3.13.16 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 113 packages in 876ms
Prepared 109 packages in 6.02s
Installed 109 packages in 409ms
 + anyio==4.15.1
 + argon2-cffi==25.1.0
 + argon2-cffi-bindings==26.1.0
 + arrow==1.4.0
 + asttokens==3.0.2
 + async-lru==2.4.0
 + attrs==26.1.0
 + babel==2.18.0
 + beautifulsoup4==4.15.0
 + bleach==6.4.0
 + certifi==2026.7.22
 + cffi==2.1.1
 + charset-normalizer==3.5.2
 + cloudpickle==3.1.2
 + comm==0.2.3
 + contourpy==1.4.0
 + cycler==0.12.1
 + debugpy==1.8.22
 + defusedxml==0.7.1
 + executing==2.3.0
 + fastjsonschema==2.22.2
 + fonttools==4.67.0
 + fqdn==1.6.0
 + h11==0.16.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + idna==3.20
 + iniconfig==2.3.1
 + ipykernel==7.3.0
 + ipython==9.17.1
 + ipython-pygments-lexers==1.1.1
 + isoduration==20.11.0
 + jedi==0.20.1
 + jinja2==3.1.6
 + joblib==1.6.0
 + json5==0.17.3
 + jsonpointer==

In [11]:
# Перезавантажуємо модуль, щоб Colab точно взяв останню версію matrices.py
import importlib
import numpy as np
import matrices

importlib.reload(matrices)


def test_get_matrix_shape_and_type():
    # Перевіряємо, що створюється NumPy-матриця правильного розміру n*m
    result = matrices.get_matrix(3, 4)

    assert isinstance(result, np.ndarray), "get_matrix має повертати np.ndarray"
    assert result.shape == (3, 4), f"Очікувалася форма (3, 4), отримано {result.shape}"


def test_add_and_immutability():
    # Перевіряємо додавання матриць та збереження вихідних даних
    x = np.array([[1.0, 2.0], [3.0, 4.0]])
    y = np.array([[5.0, 6.0], [7.0, 8.0]])
    x_before = x.copy()
    y_before = y.copy()

    result = matrices.add(x, y)

    np.testing.assert_allclose(result, [[6.0, 8.0], [10.0, 12.0]])
    np.testing.assert_allclose(x, x_before)
    np.testing.assert_allclose(y, y_before)


def test_scalar_multiplication():
    # Перевіряємо множення матриці на від'ємний та дробовий скаляр
    x = np.array([[2.0, -4.0], [6.0, 0.0]])

    result = matrices.scalar_multiplication(x, -0.5)

    np.testing.assert_allclose(result, [[-1.0, 2.0], [-3.0, 0.0]])


def test_dot_product_correctness():
    # Перевіряємо класичне матричне множення
    x = np.array([[1.0, 2.0], [3.0, 4.0]])
    y = np.array([[2.0, 0.0], [1.0, 2.0]])

    result = matrices.dot_product(x, y)

    # [[1*2 + 2*1, 1*0 + 2*2], [3*2 + 4*1, 3*0 + 4*2]] = [[4, 4], [10, 8]]
    np.testing.assert_allclose(result, [[4.0, 4.0], [10.0, 8.0]])


def test_identity_matrix_shape_and_values():
    # Перевіряємо одиничну матрицю
    result = matrices.identity_matrix(3)

    assert isinstance(result, np.ndarray), "identity_matrix має повертати np.ndarray"
    assert result.shape == (3, 3), f"Очікувалася форма (3, 3), отримано {result.shape}"
    np.testing.assert_allclose(result, np.eye(3))


def test_matrix_transpose():
    # Перевіряємо транспонування матриці (двічі транспонована дорівнює оригінальній)
    x = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])

    result = matrices.matrix_transpose(x)

    assert result.shape == (3, 2)
    np.testing.assert_allclose(result, x.T)
    np.testing.assert_allclose(matrices.matrix_transpose(result), x)


def test_hadamard_product_and_shape_check():
    # Перевіряємо покомпонентний добуток Адамара та виняток для різних розмірів
    x = np.array([[1.0, 2.0], [3.0, 4.0]])
    y = np.array([[2.0, 3.0], [4.0, 5.0]])

    result = matrices.hadamard_product(x, y)
    np.testing.assert_allclose(result, [[2.0, 6.0], [12.0, 20.0]])

    # Перевірка помилки при невідповідності розмірностей
    invalid_y = np.array([1.0, 2.0])
    try:
        matrices.hadamard_product(x, invalid_y)
        raise AssertionError("hadamard_product має викидати ValueError для різних розмірів")
    except ValueError:
        pass  # Очікувана поведінка


def test_basis_extraction():
    # Перевіряємо пошук базисних стовпців для лінійно незалежної системи
    x = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])

    result = matrices.basis(x)

    assert isinstance(result, tuple), "basis має повертати кортеж (tuple)"
    assert len(result) == 2, f"Очікувалося 2 базисних стовпці, отримано {len(result)}"


def test_norm_varieties():
    # Перевіряємо різні типи норм (Frobenius, Spectral/2, Max/Inf)
    x = np.array([[3.0, 4.0], [1.0, 2.0]])

    # Норма Фробеніуса для [[3, 4], [1, 2]] = sqrt(9+16+1+4) = sqrt(30) ≈ 5.477
    assert np.isclose(matrices.norm(x, 'fro'), np.sqrt(30.0))

    # Максимальна норма ('inf') — максимальна сума модулів елементів у рядках
    # Рядок 1: |3|+|4| = 7, Рядок 2: |1|+|2| = 3 -> Максимум 7
    assert np.isclose(matrices.norm(x, 'inf'), 7.0)


# Список усіх тестів для запуску
tests = [
    test_get_matrix_shape_and_type,
    test_add_and_immutability,
    test_scalar_multiplication,
    test_dot_product_correctness,
    test_identity_matrix_shape_and_values,
    test_matrix_transpose,
    test_hadamard_product_and_shape_check,
    test_basis_extraction,
    test_norm_varieties,
]

failed = []

for test in tests:
    try:
        test()
        print(f"PASS: {test.__name__}")
    except Exception as error:
        failed.append((test.__name__, str(error)))
        print(f"FAIL: {test.__name__}\n  {error}")

if failed:
    raise AssertionError(f"\nНе пройдено тестів: {len(failed)}")

print(f"\nУсі {len(tests)} локальних тестів пройдено.")

PASS: test_get_matrix_shape_and_type
PASS: test_add_and_immutability
PASS: test_scalar_multiplication
PASS: test_dot_product_correctness
PASS: test_identity_matrix_shape_and_values
PASS: test_matrix_transpose
PASS: test_hadamard_product_and_shape_check
PASS: test_basis_extraction
PASS: test_norm_varieties

Усі 9 локальних тестів пройдено.


# Код для matrices.pt

In [ ]:
import numpy as np
import scipy.linalg

def get_matrix(n: int, m: int) -> np.ndarray:
    """Create random matrix n * m.

    Args:
        n (int): number of rows.
        m (int): number of columns.

    Returns:
        np.ndarray: matrix n*m.
    """
    #raise NotImplementedError
    return np.random.rand(n, m)

def add(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Matrix addition.

    Args:
        x (np.ndarray): 1st matrix.
        y (np.ndarray): 2nd matrix.

    Returns:
        np.ndarray: matrix sum.
    """
    #raise NotImplementedError
    return np.add(x, y)

def scalar_multiplication(x: np.ndarray, a: float) -> np.ndarray:
    """Matrix multiplication by scalar.

    Args:
        x (np.ndarray): matrix.
        a (float): scalar.

    Returns:
        np.ndarray: multiplied matrix.
    """
    #raise NotImplementedError
    return x*a

def dot_product(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Matrices dot product.

    Args:
        x (np.ndarray): 1st matrix.
        y (np.ndarray): 2nd matrix or vector.

    Returns:
        np.ndarray: dot product.
    """
    #raise NotImplementedError
    return x @ y

def identity_matrix(dim: int) -> np.ndarray:
    """Create identity matrix with dimension `dim`.

    Args:
        dim (int): matrix dimension.

    Returns:
        np.ndarray: identity matrix.
    """
    #raise NotImplementedError
    return np.eye(dim, dtype=np.float64) # Створюємо одиничну матрицю у форматі float64 для запобігання втрати точності в подальш. обчисленнях

def matrix_inverse(x: np.ndarray) -> np.ndarray:
    """Compute inverse matrix.

    Args:
        x (np.ndarray): matrix.

    Returns:
        np.ndarray: inverse matrix.
    """
    #raise NotImplementedError
    # Обчислюємо псевдообернену матрицю для стабільності роботи з виродженими системами
    return np.linalg.pinv(x)

def matrix_transpose(x: np.ndarray) -> np.ndarray:
    """Compute transpose matrix.

    Args:
        x (np.ndarray): matrix.

    Returns:
        np.ndarray: transposed matrix.
    """
    #raise NotImplementedError
    # Транспонування через .T із незалежною копією для безпеч. тестів
    return x.T.copy()

def hadamard_product(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Compute hadamard product.

    Args:
        x (np.ndarray): 1st matrix.
        y (np.ndarray): 2nd matrix.

    Returns:
        np.ndarray: Hadamard product.
    """
    #raise NotImplementedError
    # перевіряємо строгий збіг розмірностей матриць для коректності добутку Адамара
    if x.shape != y.shape:
        raise ValueError("Матриці повинні мати однакові розміри для добутку Адамара")

    return np.multiply(x, y)

def basis(x: np.ndarray) -> tuple[int]:
    """Compute matrix basis.

    Args:
        x (np.ndarray): matrix.

    Returns:
        tuple[int]: indexes of basis columns.
    """
    #raise NotImplementedError
    # QR-розкладанням з вибором головного елемента знаходимо дійсні базисні стовпці
    _, _, pivots = scipy.linalg.qr(x.astype(np.float64, copy=False), pivoting=True)
    rank = np.linalg.matrix_rank(x)

    # Повертаємо індекси тільки тих стовпців, що увійшли до базису
    return tuple(pivots[:rank])

def norm(x: np.ndarray, order: int | float | str) -> float:
    """Matrix norm: Frobenius, Spectral or Max.

    Args:
        x (np.ndarray): vector
        order (int | float): norm's order: 'fro', 2 or inf.

    Returns:
        float: vector norm
    """
    #raise NotImplementedError
    # Для прав. роботи np.linalg.norm перетворюємо 'inf' на np.inf
    if order == 'inf':
      ord_param = np.inf
    else:
      ord_param = order

    # Обчислюємо норму з максимальною точністю float64
    return float(np.linalg.norm(x.astype(np.float64, copy=False), ord=ord_param))